# 04 — Monitoring the online endpoint

What this notebook proves to the partner:

1. **Telemetry pipeline is live.** Every `/score` request lands in
   the workspace's **Application Insights** + **Log Analytics**
   automatically — no SDK code in the scoring container required.
2. **We can answer the production questions.** Volume, p50/p95
   latency, error rate, and **per-deployment routing (blue vs green)**
   are all queryable in seconds via KQL.
3. **We can detect drift early.** A simulated drifted payload spikes
   latency / error patterns visibly in the same dashboard.
4. **There's a path to production.** AML **Model Monitoring**
   schedules + Azure Monitor alerts close the loop without
   re-architecting anything we already built.

Pre-reqs:

- Notebook `03` ran successfully — there's an `contoso-endpoint` with
  at least a `blue` deployment serving traffic.
- The workspace's Application Insights resource has its connection
  string in `.env` as `APP_INSIGHTS_CONNECTION_STRING` (auto-set when
  AML was provisioned).

## 1. Setup — workspace, endpoint, App Insights resource

Resolve everything we need from `.env` + the AML workspace metadata:

- The endpoint object (for traffic info + invoke calls).
- The **Application Insights resource ID** that AML linked at
  workspace-create time. We'll target it for KQL.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from src.config import load_settings

ENDPOINT_NAME = "contoso-endpoint"
SAMPLE_REQUEST = "../data/sample_request.json"

s = load_settings()
cred = DefaultAzureCredential()
ml = MLClient(cred, s.subscription_id, s.resource_group, s.aml_workspace)

ws = ml.workspaces.get(s.aml_workspace)
APP_INSIGHTS_ID = ws.application_insights
print("Workspace:        ", s.aml_workspace)
print("App Insights ID:  ", APP_INSIGHTS_ID)

ep = ml.online_endpoints.get(ENDPOINT_NAME)
print(f"\nEndpoint:        {ep.name}")
print(f"  scoring URI:   {ep.scoring_uri}")
print(f"  traffic split: {ep.traffic}")
print(f"  state:         {ep.provisioning_state}")

## 2. Generate traffic — normal + simulated "drift"

Two small workloads so the dashboard has something to show:

- **Normal traffic** — `N_NORMAL` calls of `data/sample_request.json` (the
  same shape the model was trained on).
- **Drifted traffic** — same payload but with `date_year=2099` and a
  bogus `countryOrRegion`. The model still returns predictions but
  this is exactly the kind of out-of-distribution input we'd want to
  catch in production.

We collect per-call latency client-side as a sanity check, but the
real story comes from the **server-side App Insights** queries in
step 3 — those are what we'd hand to a SecOps/SRE team.

In [ ]:
import copy, json, time, statistics, pathlib

N_NORMAL = 50
N_DRIFTED = 20

base_payload = json.loads(pathlib.Path(SAMPLE_REQUEST).read_text())
drifted_payload = copy.deepcopy(base_payload)
for row in drifted_payload["input_data"]["data"]:
    row[0] = "Atlantis"        # bogus countryOrRegion
    row[3] = 2099              # date_year far outside training range

tmp_dir = pathlib.Path("../data/_monitoring")
tmp_dir.mkdir(exist_ok=True)
drifted_path = tmp_dir / "sample_request_drifted.json"
drifted_path.write_text(json.dumps(drifted_payload))

def fire(label: str, n: int, request_path: str) -> list[float]:
    latencies_ms = []
    for i in range(n):
        t0 = time.perf_counter()
        try:
            ml.online_endpoints.invoke(
                endpoint_name=ENDPOINT_NAME,
                request_file=request_path,
            )
        except Exception as exc:
            print(f"[{label}] call {i+1} failed: {exc}")
            continue
        latencies_ms.append((time.perf_counter() - t0) * 1000)
    return latencies_ms

print(f"Firing {N_NORMAL} normal calls...")
normal_lat = fire("normal", N_NORMAL, SAMPLE_REQUEST)
print(f"Firing {N_DRIFTED} drifted calls...")
drifted_lat = fire("drifted", N_DRIFTED, str(drifted_path))

def stats(label, xs):
    if not xs:
        print(f"{label}: 0 successes")
        return
    xs_sorted = sorted(xs)
    p50 = xs_sorted[len(xs_sorted) // 2]
    p95 = xs_sorted[int(len(xs_sorted) * 0.95) - 1]
    print(f"{label:8} n={len(xs):3d}  p50={p50:6.0f}ms  p95={p95:6.0f}ms  mean={statistics.mean(xs):6.0f}ms")

print()
stats("normal",  normal_lat)
stats("drifted", drifted_lat)
print("\nWaiting ~60s for telemetry to land in App Insights...")
time.sleep(60)
print("Done. Proceed to step 3.")

## 3. Query App Insights with KQL

Helper that runs KQL against the workspace App Insights resource via
the `azure-monitor-query` SDK and returns a tidy `DataFrame`. We'll
reuse it for every metric.

In [ ]:
from datetime import timedelta
import pandas as pd
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

# Recover state if the kernel was restarted.
try:
    cred  # noqa: F821
except NameError:
    from azure.identity import DefaultAzureCredential
    cred = DefaultAzureCredential()

try:
    ml; s; ENDPOINT_NAME  # noqa: F821
except NameError:
    import sys, pathlib
    sys.path.append(str(pathlib.Path.cwd().parent))
    from azure.ai.ml import MLClient
    from src.config import load_settings
    s = load_settings()
    ml = MLClient(cred, s.subscription_id, s.resource_group, s.aml_workspace)
    ENDPOINT_NAME = "contoso-endpoint"

# Endpoint resource ID — diagnostic settings route AmlOnlineEndpoint*Log
# tables here, and `query_resource` against this ID hits the LA workspace
# the diagnostic settings point at (contoso-law).
ENDPOINT_RESOURCE_ID = (
    f"/subscriptions/{s.subscription_id}"
    f"/resourceGroups/{s.resource_group}"
    f"/providers/Microsoft.MachineLearningServices"
    f"/workspaces/{s.aml_workspace}"
    f"/onlineEndpoints/{ENDPOINT_NAME}"
)
print("Querying logs scoped to:\n  ", ENDPOINT_RESOURCE_ID)

logs = LogsQueryClient(cred)

def kql(query: str, lookback_min: int = 30) -> pd.DataFrame:
    res = logs.query_resource(
        resource_id=ENDPOINT_RESOURCE_ID,
        query=query,
        timespan=timedelta(minutes=lookback_min),
    )
    if res.status == LogsQueryStatus.PARTIAL:
        print("[warn] partial results:", res.partial_error)
        table = res.partial_data[0]
    else:
        table = res.tables[0]
    return pd.DataFrame(table.rows, columns=table.columns)

In [ ]:
# Sanity check: which AML log tables have data, and what does the
# TrafficLog schema actually look like in this workspace?
DISCOVERY_KQL = """
union withsource=SourceTable
    AmlOnlineEndpointTrafficLog,
    AmlOnlineEndpointConsoleLog,
    AmlOnlineEndpointEventLog
| summarize n = count() by SourceTable
| order by n desc
"""
print("Row counts (last 30 min):")
display(kql(DISCOVERY_KQL, lookback_min=30))

schema = kql("AmlOnlineEndpointTrafficLog | getschema | project ColumnName, ColumnType", lookback_min=30)
print("\nAmlOnlineEndpointTrafficLog — all columns:")
for _, r in schema.iterrows():
    print(f"  {r['ColumnName']:30s} {r['ColumnType']}")

### 3.1 Request volume + latency over time

This is the SRE go-to query: how busy is the endpoint, how fast is it
responding, and is anything failing? Bucketed at **1-minute intervals**.

In [ ]:
VOLUME_LATENCY_KQL = """
AmlOnlineEndpointTrafficLog
| summarize
    requests = count(),
    p50_ms   = percentile(TotalDurationMs, 50),
    p95_ms   = percentile(TotalDurationMs, 95),
    errors   = countif(ResponseCode >= 400)
  by bin(TimeGenerated, 1m)
| order by TimeGenerated asc
"""

df_traffic = kql(VOLUME_LATENCY_KQL, lookback_min=30)
df_traffic

In [ ]:
import matplotlib.pyplot as plt

if df_traffic.empty:
    print("No traffic logged yet. Wait another minute and re-run step 3.1.")
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ax1.bar(df_traffic["TimeGenerated"], df_traffic["requests"], width=0.0006, label="requests/min", color="#4C9AFF")
    ax1.bar(df_traffic["TimeGenerated"], df_traffic["errors"],   width=0.0006, label="errors/min",   color="#FF5630")
    ax1.set_ylabel("count / min")
    ax1.legend(loc="upper left")
    ax1.set_title(f"{ENDPOINT_NAME} — traffic + errors (last 30 min)")
    ax2.plot(df_traffic["TimeGenerated"], df_traffic["p50_ms"], marker="o", label="p50 (ms)")
    ax2.plot(df_traffic["TimeGenerated"], df_traffic["p95_ms"], marker="o", label="p95 (ms)", color="#FF8B00")
    ax2.set_ylabel("latency (ms)")
    ax2.set_xlabel("time")
    ax2.legend(loc="upper left")
    ax2.grid(alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

### 3.2 Per-deployment routing — proves blue/green is working

Group requests by `DeploymentName`. After a successful blue→green
swap from notebook `03`, **all recent traffic should land on a single
color** (whichever holds 100% traffic). If you re-run this query
during a swap, you'll see the split shifting in real time.

In [ ]:
DEPLOYMENT_KQL = """
AmlOnlineEndpointTrafficLog
| summarize requests = count(), p95_ms = percentile(TotalDurationMs, 95) by DeploymentName
| order by requests desc
"""

df_deployments = kql(DEPLOYMENT_KQL, lookback_min=30)
df_deployments

### 3.3 Top slow / failed requests

Drill-down query SREs reach for first when something looks off — the
20 slowest requests in the window with status, duration, and
deployment color. In production this is what an on-call engineer
opens after a paging alert.

In [ ]:
SLOW_REQUESTS_KQL = """
AmlOnlineEndpointTrafficLog
| project TimeGenerated, DeploymentName, ResponseCode, TotalDurationMs, ModelStatusCode, XRequestId
| order by TotalDurationMs desc
| take 20
"""

df_slow = kql(SLOW_REQUESTS_KQL, lookback_min=30)
df_slow

## 4. Path to production — what we would add next

The POC stops here, but the partner roadmap is clear:

### 4.1 Azure Monitor alerts (5 min to set up)

A scheduled-query alert against the same `requests` table — pages
on-call when error rate > 1% over 5 min, or p95 latency crosses an
SLO. Example KQL:

```kql
requests
| where cloud_RoleName has "contoso-endpoint"
| summarize
    requests = count(),
    errors   = countif(success == false),
    p95_ms   = percentile(duration, 95)
  by bin(timestamp, 5m)
| extend error_rate = todouble(errors) / requests
| where error_rate > 0.01 or p95_ms > 1000
```

Wire this to an Action Group → Teams channel / PagerDuty.

> **Bonus:** if you also enable diagnostic settings on the online
> endpoint and route them to a Log Analytics workspace, you get the
> richer `AmlOnlineEndpointTrafficLog` / `AmlOnlineEndpointConsoleLog`
> tables with explicit `EndpointName` / `DeploymentName` columns.

### 4.2 AML Model Monitoring schedule

The platform-native answer for **data drift, prediction drift, and
data quality**. Define a `MonitoringSchedule` in the AML SDK with
the training MLTable as the baseline, point it at the deployment's
data-collector outputs, and let it run nightly.

Quick sketch (not executed in this notebook — needs data collection
enabled on the deployment first):

```python
from azure.ai.ml.entities import (
    MonitoringSchedule, MonitorDefinition, MonitoringTarget,
    DataDriftSignal, PredictionDriftSignal, DataQualitySignal,
)

schedule = MonitoringSchedule(
    name="contoso-endpoint-monitor",
    trigger=...,                                # nightly
    create_monitor=MonitorDefinition(
        compute=...,
        monitoring_target=MonitoringTarget(
            ml_task="classification",
            endpoint_deployment_id=f"azureml:{ENDPOINT_NAME}:blue",
        ),
        monitoring_signals={
            "data_drift":       DataDriftSignal(...),
            "prediction_drift": PredictionDriftSignal(...),
            "data_quality":     DataQualitySignal(...),
        },
    ),
)
ml.schedules.begin_create_or_update(schedule).result()
```

Drift findings then surface in the Azure ML Studio "Monitoring" tab
and can themselves trigger alerts.

### 4.3 Application Insights workbook

The KQL queries above translate one-to-one into a sharable workbook
under the App Insights resource — partner gets a self-serve
dashboard. We'd typically build:

- Tile 1: requests/min + errors/min (bar)
- Tile 2: p50/p95 latency (line)
- Tile 3: per-deployment routing (pie)
- Tile 4: top 20 slow requests (table)
- Tile 5: drift score over time (after 4.2 lands)

### 4.4 Responsible AI considerations

For the taxi-tip model especially, before any production rollout:

- Fairness review across `paymentType` and pickup borough.
- Document the intended-use boundary (this is a sample model, not a
  pricing input).
- Add input validation so the "Atlantis" payload from step 2 is
  rejected at the gateway, not silently scored.

---

**Demo close:** the platform gave us all four telemetry pillars
(volume, latency, errors, routing) without writing any custom
instrumentation in the scoring container. Production hardening is an
incremental config exercise, not a re-architecture.

In [ ]:
kql("AmlOnlineEndpointTrafficLog | summarize n=count() by tostring(DeploymentName)", lookback_min=30)